# Trực quan hóa dữ liệu GraphicDesignEvaluation
**Data source:** https://huggingface.co/datasets/creative-graphic-design/GraphicDesignEvaluation

## 0. Cài đặt thư viện

In [ ]:
# Chạy cell này nếu chưa cài đặt
# !pip install datasets pandas matplotlib seaborn scipy

## 1. Load dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Cấu hình đồ thị ──────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})

# ── Màu sắc nhất quán ─────────────────────────────────────────
COLORS = {
    'alignment':  '#378ADD',
    'overlap':    '#1D9E75',
    'whitespace': '#D4537E',
}
PALETTE = list(COLORS.values())

# ── Load 3 datasets ───────────────────────────────────────────
BASE = "hf://datasets/creative-graphic-design/GraphicDesignEvaluation"

print("⏳ Đang load 3 datasets...")
df_alignment  = pd.read_parquet(f"{BASE}/absolute-gpt-alignment/train-00000-of-00001.parquet")
df_overlap    = pd.read_parquet(f"{BASE}/absolute-gpt-overlap/train-00000-of-00001.parquet")
df_whitespace = pd.read_parquet(f"{BASE}/absolute-gpt-whitespace/train-00000-of-00001.parquet")

# Đặt tên label rõ ràng
for df, name in [(df_alignment,'alignment'), (df_overlap,'overlap'), (df_whitespace,'whitespace')]:
    df['label'] = df['avg'].astype(float)
    df['dataset'] = name

# Gộp lại để dễ so sánh
df_all = pd.concat([df_alignment, df_overlap, df_whitespace], ignore_index=True)

print(f"   alignment  : {len(df_alignment):>4} mẫu  | label range: {df_alignment['label'].min():.1f} – {df_alignment['label'].max():.1f}")
print(f"   overlap    : {len(df_overlap):>4} mẫu  | label range: {df_overlap['label'].min():.1f} – {df_overlap['label'].max():.1f}")
print(f"   whitespace : {len(df_whitespace):>4} mẫu  | label range: {df_whitespace['label'].min():.1f} – {df_whitespace['label'].max():.1f}")
print(f"   Tổng cộng  : {len(df_all):>4} mẫu")
print("✅ Load xong!")

## 2. Thống kê mô tả (Descriptive Statistics)

In [ ]:
print("=" * 60)
print("📊 THỐNG KÊ MÔ TẢ — LABEL (avg score)")
print("=" * 60)

stats_data = []
for name, df in [('Alignment', df_alignment), ('Overlap', df_overlap), ('Whitespace', df_whitespace)]:
    s = df['label']
    stats_data.append({
        'Dataset':  name,
        'N':        len(s),
        'Min':      round(s.min(), 2),
        'Max':      round(s.max(), 2),
        'Mean':     round(s.mean(), 3),
        'Median':   round(s.median(), 3),
        'Std':      round(s.std(), 3),
        'Skewness': round(s.skew(), 3),
        'Kurtosis': round(s.kurtosis(), 3),
        'Q1':       round(s.quantile(0.25), 2),
        'Q3':       round(s.quantile(0.75), 2),
    })

stats_df = pd.DataFrame(stats_data).set_index('Dataset')
print(stats_df.to_string())
print()
print("💡 Skewness > 0: phân phối lệch phải | < 0: lệch trái")
print("💡 Kurtosis > 0: đuôi dày hơn chuẩn  | < 0: đuôi mỏng hơn")

## 3. Histogram — Phân phối nhãn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
fig.suptitle('Phân phối điểm số (Label Distribution) — 3 tập dữ liệu', fontsize=14, fontweight='bold', y=1.02)

datasets = [
    ('Alignment',  df_alignment,  COLORS['alignment']),
    ('Overlap',    df_overlap,    COLORS['overlap']),
    ('Whitespace', df_whitespace, COLORS['whitespace']),
]

for ax, (name, df, color) in zip(axes, datasets):
    data = df['label']
    # Histogram
    ax.hist(data, bins=20, color=color, alpha=0.75, edgecolor='white', linewidth=0.5)
    # Đường KDE
    kde_x = np.linspace(data.min() - 0.5, data.max() + 0.5, 200)
    kde = stats.gaussian_kde(data)
    ax2 = ax.twinx()
    ax2.plot(kde_x, kde(kde_x), color=color, linewidth=2.5, label='KDE')
    ax2.set_ylabel('Mật độ', fontsize=10, color=color)
    ax2.tick_params(axis='y', colors=color)
    ax2.spines['top'].set_visible(False)
    # Đường mean và median
    ax.axvline(data.mean(),   color='#333', linewidth=1.5, linestyle='--', label=f'Mean={data.mean():.2f}')
    ax.axvline(data.median(), color='#888', linewidth=1.5, linestyle=':',  label=f'Median={data.median():.2f}')
    ax.set_title(f'{name}', fontsize=12, fontweight='bold', color=color)
    ax.set_xlabel('Điểm số (0–10)', fontsize=10)
    ax.set_ylabel('Số lượng mẫu', fontsize=10)
    ax.legend(fontsize=9, loc='upper left')
    ax.set_xlim(0, 10)

plt.tight_layout()
plt.savefig('fig1_histogram.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Đã lưu: fig1_histogram.png")

## 4. Histogram chồng (Overlay) — So sánh 3 tập

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for name, df, color in datasets:
    data = df['label']
    ax.hist(data, bins=25, color=color, alpha=0.45, edgecolor='white', linewidth=0.3, label=name)
    kde_x = np.linspace(0, 10, 300)
    kde = stats.gaussian_kde(data)
    ax.plot([], [])  # placeholder để twinx không xung đột

# Vẽ KDE riêng trên trục y phụ
ax2 = ax.twinx()
for name, df, color in datasets:
    data = df['label']
    kde_x = np.linspace(0, 10, 300)
    kde = stats.gaussian_kde(data)
    ax2.plot(kde_x, kde(kde_x), color=color, linewidth=2.2)
ax2.set_ylabel('Mật độ KDE', fontsize=10)
ax2.spines['top'].set_visible(False)

ax.set_title('So sánh phân phối nhãn — 3 tập dữ liệu', fontsize=13, fontweight='bold')
ax.set_xlabel('Điểm số (0–10)', fontsize=11)
ax.set_ylabel('Số lượng mẫu', fontsize=11)
ax.set_xlim(0, 10)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('fig2_overlay_histogram.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Đã lưu: fig2_overlay_histogram.png")

## 5. Box Plot — Phân vị và ngoại lệ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Box plot ──────────────────────────────────────────────────
ax = axes[0]
box_data  = [df_alignment['label'], df_overlap['label'], df_whitespace['label']]
box_names = ['Alignment', 'Overlap', 'Whitespace']

bp = ax.boxplot(box_data, patch_artist=True, notch=True,
                medianprops=dict(color='white', linewidth=2.5),
                whiskerprops=dict(linewidth=1.5),
                capprops=dict(linewidth=2),
                flierprops=dict(marker='o', markersize=4, alpha=0.5))

for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
for flier, color in zip(bp['fliers'], PALETTE):
    flier.set_markerfacecolor(color)

ax.set_xticklabels(box_names, fontsize=11)
ax.set_ylabel('Điểm số (0–10)', fontsize=11)
ax.set_title('Box Plot — Phân vị điểm số', fontsize=12, fontweight='bold')
ax.set_ylim(0, 11)

# Annotate mean
for i, (df, color) in enumerate([(df_alignment, COLORS['alignment']),
                                   (df_overlap,   COLORS['overlap']),
                                   (df_whitespace, COLORS['whitespace'])], 1):
    mean_val = df['label'].mean()
    ax.plot(i, mean_val, 'D', color=color, markersize=8, zorder=5,
            markeredgecolor='white', markeredgewidth=1.5, label='Mean' if i==1 else '')
    ax.text(i + 0.25, mean_val, f'{mean_val:.2f}', va='center', fontsize=9, color='#444')
ax.legend(fontsize=9)

# ── Violin plot ───────────────────────────────────────────────
ax = axes[1]
vp = ax.violinplot(box_data, positions=[1, 2, 3], showmedians=True, showextrema=True)

for body, color in zip(vp['bodies'], PALETTE):
    body.set_facecolor(color)
    body.set_alpha(0.7)
vp['cmedians'].set_color('white')
vp['cmedians'].set_linewidth(2)
vp['cmaxes'].set_color('#555')
vp['cmins'].set_color('#555')
vp['cbars'].set_color('#555')

# Scatter điểm thực tế (jitter)
for i, (df, color) in enumerate([(df_alignment, COLORS['alignment']),
                                   (df_overlap,   COLORS['overlap']),
                                   (df_whitespace, COLORS['whitespace'])], 1):
    jitter = np.random.normal(i, 0.06, size=len(df))
    ax.scatter(jitter, df['label'], alpha=0.15, s=6, color=color, zorder=2)

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(box_names, fontsize=11)
ax.set_ylabel('Điểm số (0–10)', fontsize=11)
ax.set_title('Violin Plot — Phân phối chi tiết', fontsize=12, fontweight='bold')
ax.set_ylim(0, 11)

plt.tight_layout()
plt.savefig('fig3_boxplot_violin.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Đã lưu: fig3_boxplot_violin.png")

## 6. Scatter Plot — Tương quan giữa các tập

In [ ]:
# Merge 3 tập theo index (cùng ảnh thiết kế)
df_merged = pd.DataFrame({
    'alignment':  df_alignment['label'].values,
    'overlap':    df_overlap['label'].values,
    'whitespace': df_whitespace['label'].values,
})

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Scatter Plot — Tương quan điểm số giữa các tiêu chí', fontsize=13, fontweight='bold', y=1.02)

pairs = [
    ('alignment',  'overlap',    COLORS['alignment'],  'Alignment vs Overlap'),
    ('alignment',  'whitespace', COLORS['whitespace'], 'Alignment vs Whitespace'),
    ('overlap',    'whitespace', COLORS['overlap'],    'Overlap vs Whitespace'),
]

for ax, (x_col, y_col, color, title) in zip(axes, pairs):
    x = df_merged[x_col]
    y = df_merged[y_col]

    ax.scatter(x, y, alpha=0.35, s=20, color=color, edgecolors='none')

    # Đường hồi quy
    slope, intercept, r, p, _ = stats.linregress(x, y)
    line_x = np.linspace(x.min(), x.max(), 100)
    ax.plot(line_x, slope * line_x + intercept, 'k--', linewidth=1.8, label=f'r = {r:.3f}')

    # Diagonal reference
    lim = [min(x.min(), y.min()) - 0.3, max(x.max(), y.max()) + 0.3]
    ax.plot(lim, lim, color='#aaa', linewidth=1, linestyle=':', label='y = x')

    p_str = f'p < 0.001' if p < 0.001 else f'p = {p:.3f}'
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel(x_col.capitalize(), fontsize=10)
    ax.set_ylabel(y_col.capitalize(), fontsize=10)
    ax.legend(fontsize=9)
    ax.text(0.05, 0.92, p_str, transform=ax.transAxes, fontsize=9, color='#555')
    ax.set_xlim(lim); ax.set_ylim(lim)

plt.tight_layout()
plt.savefig('fig4_scatter.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Đã lưu: fig4_scatter.png")

## 7. Correlation Matrix — Ma trận tương quan

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Pearson correlation ───────────────────────────────────────
ax = axes[0]
corr_pearson = df_merged.corr(method='pearson')

mask = np.triu(np.ones_like(corr_pearson, dtype=bool), k=1)
sns.heatmap(
    corr_pearson, ax=ax,
    annot=True, fmt='.3f', annot_kws={'size': 13, 'weight': 'bold'},
    cmap='RdBu_r', vmin=-1, vmax=1, center=0,
    square=True, linewidths=2, linecolor='white',
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Pearson Correlation Matrix', fontsize=12, fontweight='bold')
ax.set_xticklabels(['Alignment', 'Overlap', 'Whitespace'], fontsize=10)
ax.set_yticklabels(['Alignment', 'Overlap', 'Whitespace'], fontsize=10, rotation=0)

# ── Spearman correlation ──────────────────────────────────────
ax = axes[1]
corr_spearman = df_merged.corr(method='spearman')

sns.heatmap(
    corr_spearman, ax=ax,
    annot=True, fmt='.3f', annot_kws={'size': 13, 'weight': 'bold'},
    cmap='RdBu_r', vmin=-1, vmax=1, center=0,
    square=True, linewidths=2, linecolor='white',
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Spearman Correlation Matrix', fontsize=12, fontweight='bold')
ax.set_xticklabels(['Alignment', 'Overlap', 'Whitespace'], fontsize=10)
ax.set_yticklabels(['Alignment', 'Overlap', 'Whitespace'], fontsize=10, rotation=0)

fig.suptitle('Ma trận tương quan — Điểm số giữa 3 tiêu chí', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig5_correlation_matrix.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Đã lưu: fig5_correlation_matrix.png")

## 8. Phân tích phân phối điểm số theo dải (Score Band Analysis)

In [ ]:
def score_band(score):
    if score >= 8:   return '🟢 Xuất sắc (8–10)'
    elif score >= 6: return '🟡 Tốt (6–8)'
    elif score >= 4: return '🟠 Trung bình (4–6)'
    else:            return '🔴 Kém (0–4)'

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Phân bố mẫu theo dải chất lượng — 3 tập dữ liệu', fontsize=13, fontweight='bold', y=1.02)

band_colors = {
    '🟢 Xuất sắc (8–10)':    '#2ecc71',
    '🟡 Tốt (6–8)':          '#f39c12',
    '🟠 Trung bình (4–6)':   '#e67e22',
    '🔴 Kém (0–4)':          '#e74c3c',
}

for ax, (name, df, color) in zip(axes, datasets):
    df = df.copy()
    df['band'] = df['label'].apply(score_band)
    counts = df['band'].value_counts().reindex(list(band_colors.keys()), fill_value=0)
    pct = counts / counts.sum() * 100

    bars = ax.barh(counts.index, counts.values,
                   color=[band_colors[b] for b in counts.index],
                   edgecolor='white', linewidth=0.5, height=0.6)

    # Annotate percentage
    for bar, p in zip(bars, pct.values):
        ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                f'{p:.1f}%', va='center', fontsize=10, color='#444')

    ax.set_title(name, fontsize=12, fontweight='bold', color=color)
    ax.set_xlabel('Số lượng mẫu', fontsize=10)
    ax.set_xlim(0, counts.max() + 40)
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig('fig6_score_bands.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Đã lưu: fig6_score_bands.png")

## 9. Kiểm định phân phối chuẩn (Normality Test)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Q-Q Plot — Kiểm tra phân phối chuẩn', fontsize=13, fontweight='bold', y=1.02)

print("=" * 55)
print("🔬 KIỂM ĐỊNH SHAPIRO-WILK (H₀: phân phối chuẩn)")
print("=" * 55)

for ax, (name, df, color) in zip(axes, datasets):
    data = df['label'].values
    # Q-Q plot
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist='norm')
    ax.scatter(osm, osr, alpha=0.5, s=15, color=color, edgecolors='none')
    line_x = np.array([osm.min(), osm.max()])
    ax.plot(line_x, slope * line_x + intercept, 'k--', linewidth=1.8)
    ax.set_title(name, fontsize=11, fontweight='bold', color=color)
    ax.set_xlabel('Lý thuyết (chuẩn)', fontsize=10)
    ax.set_ylabel('Thực tế', fontsize=10)
    ax.text(0.05, 0.92, f'r = {r:.3f}', transform=ax.transAxes, fontsize=9)

    # Shapiro-Wilk test (dùng mẫu 200 để tránh cảnh báo)
    sample = np.random.choice(data, size=min(200, len(data)), replace=False)
    stat, p = stats.shapiro(sample)
    conclusion = '❌ Không chuẩn' if p < 0.05 else '✅ Có thể chuẩn'
    print(f"   {name:<12}: W={stat:.4f}, p={p:.4f}  → {conclusion}")

print("\n💡 p < 0.05 → bác bỏ H₀ → phân phối KHÔNG chuẩn → dùng Spearman thay Pearson")

plt.tight_layout()
plt.savefig('fig7_qqplot.png', bbox_inches='tight', dpi=150)
plt.show()
print("\n✅ Đã lưu: fig7_qqplot.png")

## 10. Xem mẫu ảnh thiết kế theo dải điểm (Sample Images)

In [ ]:
import io
from PIL import Image

def load_image(img_data):
    """Load ảnh từ các định dạng khác nhau trong dataset"""
    if isinstance(img_data, dict):
        if img_data.get('bytes'):
            return Image.open(io.BytesIO(img_data['bytes'])).convert('RGB')
        elif img_data.get('path'):
            return Image.open(img_data['path']).convert('RGB')
    elif isinstance(img_data, bytes):
        return Image.open(io.BytesIO(img_data)).convert('RGB')
    elif hasattr(img_data, 'convert'):
        return img_data.convert('RGB')
    return None

# Hiển thị ảnh từ df_alignment theo 4 dải điểm
df_align = df_alignment.copy()
df_align['band_label'] = pd.cut(
    df_align['label'],
    bins=[0, 4, 6, 8, 10],
    labels=['Kém (0–4)', 'Trung bình (4–6)', 'Tốt (6–8)', 'Xuất sắc (8–10)']
)

band_palette = {
    'Kém (0–4)':          '#e74c3c',
    'Trung bình (4–6)':   '#e67e22',
    'Tốt (6–8)':          '#f39c12',
    'Xuất sắc (8–10)':    '#2ecc71',
}

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle('Mẫu ảnh thiết kế — Alignment dataset (theo dải điểm)', fontsize=14, fontweight='bold')

for row_i, (band_name, band_color) in enumerate(band_palette.items()):
    subset = df_align[df_align['band_label'] == band_name]
    samples = subset.sample(min(4, len(subset)), random_state=42)

    for col_i in range(4):
        ax = axes[row_i][col_i]
        if col_i < len(samples):
            row = samples.iloc[col_i]
            img = load_image(row['image'])
            if img:
                ax.imshow(img)
                score = row['label']
                ax.set_title(f'{score:.1f}', fontsize=10, color=band_color, fontweight='bold')
            else:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
        else:
            ax.text(0.5, 0.5, '—', ha='center', va='center', transform=ax.transAxes, color='#aaa')

        ax.axis('off')
        if col_i == 0:
            ax.set_ylabel(band_name, fontsize=9, color=band_color,
                          rotation=90, labelpad=5)
            ax.yaxis.set_label_position('left')
            ax.axis('on')
            ax.set_yticks([]); ax.set_xticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

# Nhãn hàng
for row_i, (band_name, band_color) in enumerate(band_palette.items()):
    fig.text(0.01, 0.87 - row_i * 0.23, band_name,
             fontsize=10, color=band_color, fontweight='bold',
             va='center', rotation=90)

plt.tight_layout(rect=[0.03, 0, 1, 0.97])
plt.savefig('fig8_sample_images.png', bbox_inches='tight', dpi=120)
plt.show()
print("✅ Đã lưu: fig8_sample_images.png")

## 11. Tổng hợp — Dashboard EDA toàn bộ

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle('EDA Dashboard — GraphicDesignEvaluation Dataset', fontsize=16, fontweight='bold', y=0.98)

# Layout: 3 hàng
gs = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.35)

# ── Hàng 1: Histogram x3 + Bar thống kê ─────────────────────
for col_i, (name, df, color) in enumerate(datasets):
    ax = fig.add_subplot(gs[0, col_i])
    data = df['label']
    ax.hist(data, bins=18, color=color, alpha=0.8, edgecolor='white', linewidth=0.4)
    ax.axvline(data.mean(), color='#333', lw=1.5, ls='--')
    ax.set_title(f'{name}\n(n=400)', fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('Điểm', fontsize=9); ax.set_ylabel('Số mẫu', fontsize=9)
    ax.set_xlim(0, 10)
    ax.tick_params(labelsize=8)
    ax.text(0.98, 0.95, f'μ={data.mean():.2f}\nσ={data.std():.2f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

# Bar thống kê tổng hợp
ax4 = fig.add_subplot(gs[0, 3])
names_short = ['Align.', 'Overlap', 'Wspace']
means   = [df_alignment['label'].mean(), df_overlap['label'].mean(), df_whitespace['label'].mean()]
stds    = [df_alignment['label'].std(),  df_overlap['label'].std(),  df_whitespace['label'].std()]
x_pos = np.arange(3)
bars = ax4.bar(x_pos, means, yerr=stds, capsize=5, color=PALETTE, alpha=0.85,
               edgecolor='white', linewidth=0.5, error_kw={'ecolor':'#555','elinewidth':1.5})
ax4.set_xticks(x_pos); ax4.set_xticklabels(names_short, fontsize=9)
ax4.set_ylabel('Mean ± Std', fontsize=9); ax4.set_ylim(0, 9)
ax4.set_title('Mean ± Std', fontsize=10, fontweight='bold')
ax4.tick_params(labelsize=8)
for bar, mean in zip(bars, means):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
             f'{mean:.2f}', ha='center', fontsize=8)

# ── Hàng 2: Box plot + Scatter + Correlation ─────────────────
ax5 = fig.add_subplot(gs[1, :2])
bp = ax5.boxplot([df_alignment['label'], df_overlap['label'], df_whitespace['label']],
                  patch_artist=True, notch=True,
                  medianprops=dict(color='white', lw=2))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color); patch.set_alpha(0.8)
ax5.set_xticklabels(['Alignment', 'Overlap', 'Whitespace'], fontsize=10)
ax5.set_ylabel('Điểm số', fontsize=10)
ax5.set_title('Box Plot', fontsize=11, fontweight='bold')

ax6 = fig.add_subplot(gs[1, 2])
ax6.scatter(df_merged['alignment'], df_merged['overlap'],
            alpha=0.3, s=12, color=COLORS['alignment'], edgecolors='none')
slope, intercept, r, _, _ = stats.linregress(df_merged['alignment'], df_merged['overlap'])
lx = np.linspace(df_merged['alignment'].min(), df_merged['alignment'].max(), 100)
ax6.plot(lx, slope*lx+intercept, 'k--', lw=1.5)
ax6.set_xlabel('Alignment', fontsize=9); ax6.set_ylabel('Overlap', fontsize=9)
ax6.set_title(f'Align. vs Overlap\nr={r:.3f}', fontsize=10, fontweight='bold')
ax6.tick_params(labelsize=8)

ax7 = fig.add_subplot(gs[1, 3])
corr = df_merged.corr(method='pearson')
sns.heatmap(corr, ax=ax7, annot=True, fmt='.2f', annot_kws={'size':11,'weight':'bold'},
            cmap='RdBu_r', vmin=-1, vmax=1, center=0,
            square=True, linewidths=2, linecolor='white',
            xticklabels=['Align','Overlap','Wspace'],
            yticklabels=['Align','Overlap','Wspace'],
            cbar=False)
ax7.set_title('Pearson Correlation', fontsize=10, fontweight='bold')
ax7.tick_params(labelsize=8)

# ── Hàng 3: Score band x3 + pie tổng ─────────────────────────
band_names = ['Kém\n(0–4)', 'T.Bình\n(4–6)', 'Tốt\n(6–8)', 'Xuất sắc\n(8–10)']
band_cols  = ['#e74c3c', '#e67e22', '#f39c12', '#2ecc71']

all_counts = [0, 0, 0, 0]
for col_i, (name, df, color) in enumerate(datasets):
    ax = fig.add_subplot(gs[2, col_i])
    counts = [
        (df['label'] < 4).sum(),
        ((df['label'] >= 4) & (df['label'] < 6)).sum(),
        ((df['label'] >= 6) & (df['label'] < 8)).sum(),
        (df['label'] >= 8).sum(),
    ]
    all_counts = [a+b for a,b in zip(all_counts, counts)]
    bars = ax.bar(band_names, counts, color=band_cols, edgecolor='white', linewidth=0.5)
    for bar, cnt in zip(bars, counts):
        pct = cnt / len(df) * 100
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{pct:.0f}%', ha='center', fontsize=8)
    ax.set_title(name, fontsize=10, fontweight='bold', color=color)
    ax.set_ylabel('Số mẫu', fontsize=9); ax.tick_params(labelsize=8)

ax_pie = fig.add_subplot(gs[2, 3])
ax_pie.pie(all_counts, labels=band_names, colors=band_cols, autopct='%1.0f%%',
           startangle=90, pctdistance=0.75,
           textprops={'fontsize': 8},
           wedgeprops={'edgecolor':'white','linewidth':1.5})
ax_pie.set_title('Tổng 3 tập', fontsize=10, fontweight='bold')

plt.savefig('fig9_dashboard_full.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Đã lưu: fig9_dashboard_full.png")
print("\n🎉 Hoàn tất toàn bộ EDA!")